# 01. Data Creation Notebook

Prepare COCO access, fixed splits, synthetic shape generation, and automatic label creation.

In [ ]:
import hashlib
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Literal

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageDraw, ImageFilter
from torchvision.datasets import CocoDetection
import matplotlib.pyplot as plt

print("PyTorch:", torch.__version__)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd()
KAGGLE_INPUT_ROOT = Path("/kaggle/input")
COCO_SYMLINK = Path("data/coco")


def find_kaggle_coco_root() -> Path | None:
    direct_candidates = [
        Path("/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017"),
        Path("/kaggle/input/coco-2017-dataset/coco2017"),
    ]
    for candidate in direct_candidates:
        if (candidate / "train2017").exists() and (candidate / "val2017").exists() and (candidate / "annotations").exists():
            return candidate

    if KAGGLE_INPUT_ROOT.exists():
        for candidate in sorted(KAGGLE_INPUT_ROOT.glob("**/coco2017")):
            if (candidate / "train2017").exists() and (candidate / "val2017").exists() and (candidate / "annotations").exists():
                return candidate

    return None


kaggle_coco_root = find_kaggle_coco_root()
if kaggle_coco_root is not None and not COCO_SYMLINK.exists():
    COCO_SYMLINK.parent.mkdir(parents=True, exist_ok=True)
    os.symlink(kaggle_coco_root, COCO_SYMLINK, target_is_directory=True)
    print(f"Created symlink: {COCO_SYMLINK} -> {kaggle_coco_root}")
elif COCO_SYMLINK.exists():
    print(f"Using existing COCO path: {COCO_SYMLINK.resolve()}")
else:
    print("No Kaggle COCO dataset detected automatically. Make sure data/coco exists.")


In [ ]:
# Create the project directories used during setup, training, and evaluation.
for path in [
    Path("data"),
    Path("results"),
    Path("figures"),
    Path("figures/generated_samples"),
    Path("figures/prediction_visualizations"),
]:
    path.mkdir(parents=True, exist_ok=True)

print("Directory setup completed.")

### Kaggle Note

This notebook can run directly on Kaggle if the COCO 2017 dataset is attached as an input.  
The setup cell above creates a `data/coco` symlink automatically when it detects a Kaggle COCO input.


---
## 2. Dataset and Split

In [ ]:
# Assignment constants
GLOBAL_SEED          = 2025
TRAIN_SIZE           = 5000   # first 5000 images from train2017
VAL_SIZE             = 1000   # first 1000 images from val2017
TEST_SIZE            = 1000   # next  1000 images from val2017 (indices 1000–1999)
POSITIVE_RATIO       = 0.70   # 70% positive, 30% negative
MAX_SHAPES_PER_IMAGE = 3      # 1 to 3 shapes per positive image


# Deterministic seed
# Do NOT use Python's built-in hash(); output varies between sessions.
def make_seed(split_name: str, image_id: int, global_seed: int = GLOBAL_SEED) -> int:
    key = f"{split_name}_{image_id}_{global_seed}".encode("utf-8")
    return int(hashlib.sha256(key).hexdigest()[:8], 16)


# COCO base initialization
# Required directory structure:
#   data/coco/
#   ├── train2017/
#   ├── val2017/
#   └── annotations/
#       ├── instances_train2017.json
#       └── instances_val2017.json
def build_coco_bases(data_root: str | Path = "data/coco"):
    root = Path(data_root)

    train_base = CocoDetection(
        root=str(root / "train2017"),
        annFile=str(root / "annotations" / "instances_train2017.json"),
    )

    val_base = CocoDetection(
        root=str(root / "val2017"),
        annFile=str(root / "annotations" / "instances_val2017.json"),
    )

    return train_base, val_base


# Fixed split protocol
#   - Sort all image IDs in increasing order
#   - train : first 5000 from train2017
#   - val   : first 1000 from val2017
#   - test  : next  1000 from val2017  (do NOT use for model selection)
def get_split_ids(
    coco_base: CocoDetection,
    split: Literal["train", "val", "test"],
) -> list[int]:
    all_ids = sorted(coco_base.coco.getImgIds())

    if split == "train":
        return all_ids[:TRAIN_SIZE]
    elif split == "val":
        return all_ids[:VAL_SIZE]
    elif split == "test":
        return all_ids[VAL_SIZE : VAL_SIZE + TEST_SIZE]
    else:
        raise ValueError(f"Unknown split: {split!r}")

In [ ]:
train_base, val_base = build_coco_bases("data/coco")

train_ids = get_split_ids(train_base, "train")   # first 5000 from train2017
val_ids   = get_split_ids(val_base,   "val")     # first 1000 from val2017
test_ids  = get_split_ids(val_base,   "test")    # next  1000 from val2017

print(f"Train: {len(train_ids)}  Val: {len(val_ids)}  Test: {len(test_ids)}")

---
## 3. Synthetic Shape Generation

Synthetic samples are created by drawing random shapes directly onto COCO background images. The current generator supports rectangles, ellipses, and triangles, and varies location, size, color, opacity, and the number of shapes per image.

Positive images contain between **1 and 3** synthetic shapes, while negative images contain no target shape; the positive/negative ratio is approximately **70/30** as suggested in the assignment.

To keep the task nontrivial, the generator already includes multiple difficulty mechanisms such as random opacity, low-contrast colors sampled from local image content, Gaussian blur on inserted shapes, and additive noise on the final image.

Part of the synthetic shape placement and bounding-box design was inspired by prior experience from a Teknofest Aviation AI project. However, for this assignment, all synthetic shapes and their corresponding labels are generated fully automatically, and no manual annotation is used.

In [ ]:
@dataclass
class ShapeGenerationConfig:
    opacity_range: tuple[int, int] = (96, 220)
    blur_range: tuple[float, float] = (0.2, 1.6)
    noise_std_range: tuple[float, float] = (2.0, 8.0)
    color_jitter: int = 35
    min_size_ratio: float = 1 / 12
    max_size_ratio: float = 1 / 4


DEFAULT_SHAPE_CONFIG = ShapeGenerationConfig()


class ShapeGenerator:
    def __init__(self, config: ShapeGenerationConfig | None = None):
        self.config = config or DEFAULT_SHAPE_CONFIG

    def _sample_color(self, image_np: np.ndarray, x1: int, y1: int, x2: int, y2: int, rng: random.Random):
        patch = image_np[y1:y2, x1:x2]
        if patch.size == 0:
            patch = image_np

        base = patch.reshape(-1, 3).mean(axis=0)
        jitter = np.array([
            rng.randint(-self.config.color_jitter, self.config.color_jitter) for _ in range(3)
        ], dtype=np.float32)
        color = np.clip(base + jitter, 0, 255).astype(np.uint8)
        alpha = rng.randint(*self.config.opacity_range)
        return tuple(int(v) for v in color), alpha

    def _draw_shape(self, draw, shape_type: str, box, fill):
        x1, y1, x2, y2 = box
        if shape_type == "rectangle":
            draw.rectangle(box, fill=fill)
        elif shape_type == "ellipse":
            draw.ellipse(box, fill=fill)
        elif shape_type == "triangle":
            pts = [((x1 + x2) / 2, y1), (x1, y2), (x2, y2)]
            draw.polygon(pts, fill=fill)
        else:
            raise ValueError(f"Unsupported shape type: {shape_type}")

    def generate(self, image, n_shapes: int, rng: random.Random):
        """
        Draw n_shapes synthetic shapes onto image.

        Args:
            image    : PIL.Image (RGB)
            n_shapes : number of shapes to draw (0 for negative images)
            rng      : seeded random.Random instance

        Returns:
            augmented_image : PIL.Image
            boxes           : list[list[float]]   [[x1, y1, x2, y2], ...]
            mask            : np.ndarray [H, W]   1=shape pixel, 0=background
        """
        image = image.convert("RGB")
        image_np = np.array(image)
        h, w = image_np.shape[:2]

        canvas = image.convert("RGBA")
        binary_mask = np.zeros((h, w), dtype=np.uint8)
        boxes = []
        shape_types = ["rectangle", "ellipse", "triangle"]

        min_w = max(12, int(w * self.config.min_size_ratio))
        max_w = max(min_w + 1, int(w * self.config.max_size_ratio))
        min_h = max(12, int(h * self.config.min_size_ratio))
        max_h = max(min_h + 1, int(h * self.config.max_size_ratio))

        for _ in range(n_shapes):
            box_w = rng.randint(min_w, max_w)
            box_h = rng.randint(min_h, max_h)
            x1 = rng.randint(0, max(0, w - box_w - 1))
            y1 = rng.randint(0, max(0, h - box_h - 1))
            x2 = min(w - 1, x1 + box_w)
            y2 = min(h - 1, y1 + box_h)

            color_rgb, alpha = self._sample_color(image_np, x1, y1, x2, y2, rng)
            fill_rgba = (*color_rgb, alpha)
            shape_type = rng.choice(shape_types)
            box = (x1, y1, x2, y2)

            overlay = Image.new("RGBA", (w, h), (0, 0, 0, 0))
            overlay_draw = ImageDraw.Draw(overlay)
            self._draw_shape(overlay_draw, shape_type, box, fill_rgba)

            blur_radius = rng.uniform(*self.config.blur_range)
            overlay = overlay.filter(ImageFilter.GaussianBlur(radius=blur_radius))
            canvas = Image.alpha_composite(canvas, overlay)

            shape_mask = Image.new("L", (w, h), 0)
            mask_draw = ImageDraw.Draw(shape_mask)
            self._draw_shape(mask_draw, shape_type, box, 1)
            binary_mask = np.maximum(binary_mask, np.array(shape_mask, dtype=np.uint8))
            boxes.append([float(x1), float(y1), float(x2), float(y2)])

        augmented = np.array(canvas.convert("RGB"), dtype=np.float32)
        noise_std = rng.uniform(*self.config.noise_std_range)
        np_rng = np.random.default_rng(rng.randint(0, 2**32 - 1))
        augmented = np.clip(augmented + np_rng.normal(0.0, noise_std, augmented.shape), 0, 255).astype(np.uint8)
        augmented_image = Image.fromarray(augmented)

        return augmented_image, boxes, binary_mask


In [ ]:
# Visualize at least 12 generated examples (§4).
# Must include both positive and negative samples.

def show_generated_samples(n_positive: int = 6, n_negative: int = 6, save_dir: str | Path = "figures/generated_samples"):
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    generator = ShapeGenerator(DEFAULT_SHAPE_CONFIG)
    examples = []

    for idx in range(n_positive):
        image_id = val_ids[idx]
        coco_idx = val_base.ids.index(image_id)
        image, _ = val_base[coco_idx]
        rng = random.Random(make_seed("val", image_id))
        n_shapes = rng.randint(1, MAX_SHAPES_PER_IMAGE)
        augmented, boxes, _ = generator.generate(image, n_shapes, rng)
        examples.append((augmented, boxes, f"positive | shapes={len(boxes)}"))

    for idx in range(n_negative):
        image_id = val_ids[n_positive + idx]
        coco_idx = val_base.ids.index(image_id)
        image, _ = val_base[coco_idx]
        rng = random.Random(make_seed("val", image_id))
        augmented, boxes, _ = generator.generate(image, 0, rng)
        examples.append((augmented, boxes, "negative | shapes=0"))

    rows = 3
    cols = 4
    fig, axes = plt.subplots(rows, cols, figsize=(16, 12))
    axes = axes.flatten()

    for ax, (image, boxes, title) in zip(axes, examples):
        ax.imshow(image)
        for x1, y1, x2, y2 in boxes:
            ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="lime", linewidth=2))
        ax.set_title(title)
        ax.axis("off")

    for ax in axes[len(examples):]:
        ax.axis("off")

    fig.tight_layout()
    save_path = save_dir / "generated_samples_grid.png"
    fig.savefig(save_path, dpi=160, bbox_inches="tight")
    plt.show()
    print(f"Saved generated sample grid to {save_path}")


show_generated_samples()

---
## 4. Label Generation

Choose **Option A** (Object Detection) or **Option B** (Semantic Segmentation).

A classification-only model is **not sufficient** for full credit (§5).

In [ ]:
# PyTorch Dataset wrapper
#
# The wrapper must (§2):
#   1. Load a COCO image
#   2. Add one or more synthetic shapes
#   3. Generate the corresponding target label automatically
#   4. Return the modified image and generated target
#
# Target format — Option A: Object Detection (§5)
#   target = {
#       "boxes":       FloatTensor[N, 4]   # x_min, y_min, x_max, y_max
#       "labels":      LongTensor[N]        # all synthetic shapes share label 1
#       "image_id":    int
#       "is_positive": bool
#   }
#   If negative: boxes = zeros((0,4)), labels = zeros((0,))
#
# Target format — Option B: Semantic Segmentation (§5)
#   target = {
#       "mask":        LongTensor or FloatTensor[H, W]   # 1=shape, 0=background
#       "image_id":    int
#       "is_positive": bool
#   }
#   If negative: mask contains only zeros
class SyntheticShapeDataset(Dataset):
    def __init__(
        self,
        coco_base: CocoDetection,
        image_ids: list[int],
        split: Literal["train", "val", "test"],
        task: Literal["detection", "segmentation"],
        transform=None,
        generator_config: ShapeGenerationConfig | None = None,
        positive_ratio: float = POSITIVE_RATIO,
        max_shapes_per_image: int = MAX_SHAPES_PER_IMAGE,
    ):
        self.coco_base = coco_base
        self.image_ids = image_ids
        self.split     = split
        self.task      = task
        self.transform = transform
        self.generator = ShapeGenerator(generator_config)
        self.positive_ratio = positive_ratio
        self.max_shapes_per_image = max_shapes_per_image
        self.id_to_index = {img_id: i for i, img_id in enumerate(self.coco_base.ids)}

    def __len__(self) -> int:
        return len(self.image_ids)

    def __getitem__(self, idx: int):
        image_id = self.image_ids[idx]

        # Seed: random for train, deterministic for val/test (§3)
        if self.split == "train":
            rng = random.Random()
        else:
            rng = random.Random(make_seed(self.split, image_id))

        # Positive / negative decision — 70% positive, 30% negative (§4)
        is_positive = rng.random() < self.positive_ratio
        n_shapes    = rng.randint(1, self.max_shapes_per_image) if is_positive else 0

        coco_idx = self.id_to_index[image_id]
        image, _ = self.coco_base[coco_idx]
        augmented_image, boxes, mask = self.generator.generate(image, n_shapes, rng)

        boxes_tensor = torch.tensor(boxes, dtype=torch.float32)
        if len(boxes) == 0:
            boxes_tensor = torch.zeros((0, 4), dtype=torch.float32)
            labels_tensor = torch.zeros((0,), dtype=torch.long)
        else:
            labels_tensor = torch.ones((len(boxes),), dtype=torch.long)

        if self.transform is not None:
            augmented_image = self.transform(augmented_image)
        else:
            augmented_image = T.ToTensor()(augmented_image)

        if self.task == "detection":
            target = {
                "boxes": boxes_tensor,
                "labels": labels_tensor,
                "image_id": image_id,
                "is_positive": is_positive,
            }
        else:
            target = {
                "mask": torch.as_tensor(mask, dtype=torch.long),
                "image_id": image_id,
                "is_positive": is_positive,
            }

        return augmented_image, target

In [ ]:
import zipfile

EXPORT_DIR = Path("figures/generated_samples")
EXPORT_ZIP = Path("figures/generated_samples.zip")

with zipfile.ZipFile(EXPORT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for file_path in sorted(EXPORT_DIR.glob("**/*")):
        if file_path.is_file():
            archive.write(file_path, file_path.relative_to(EXPORT_DIR.parent))

print(f"Packed generated sample files into {EXPORT_ZIP}")
